# PDD Mobility Scanner — Trail Object Detection

Detect benches, chairs, and other trailside amenities in body-camera photos using YOLOv8, optionally classify them with a custom **Teachable Machine** model, match detections to GPS coordinates from a companion sensor CSV, and export a CSV ready for map visualization.

Pipeline:
1. Upload `.jpg` images, `trail_data.csv`, and optionally a Teachable Machine model (`keras_model.h5` + `labels.txt`)
2. Run **YOLOv8** object detection on every image (bounding boxes)
3. If a Teachable Machine model is provided, classify each YOLO crop with it (e.g. "green_park_bench", "memorial_bench")
4. Match each image to its GPS waypoint from the CSV
5. Export `trail_detections.csv` with detection + location data

**GPU runtime recommended** (Runtime → Change runtime type → T4 GPU) but CPU works too — inference will just be slower.

### Teachable Machine setup (optional)

Train a model at [teachablemachine.withgoogle.com](https://teachablemachine.withgoogle.com/):
1. Create an **Image Project**
2. Add classes for the specific amenities in your park (e.g. photos of your park's bench style, trash cans, signs)
3. Train and export as **TensorFlow → Keras** — you'll get `keras_model.h5` and `labels.txt`
4. Upload both files alongside your images and CSV

## Step 1: Install dependencies

In [ ]:
!pip install -q ultralytics pandas pillow tensorflow

## Step 2: Upload images, CSV, and (optional) Teachable Machine model

Upload all files in one batch:
- `.jpg` / `.jpeg` / `.png` — trail images
- `.csv` — trail_data.csv with GPS + sensor data
- `keras_model.h5` + `labels.txt` — Teachable Machine export (optional)

In [ ]:
from google.colab import files

uploaded = files.upload()

# Separate files by type
csv_files = [f for f in sorted(uploaded.keys()) if f.lower().endswith('.csv')]
image_files = [f for f in sorted(uploaded.keys()) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
tm_model_file = [f for f in uploaded.keys() if f.lower().endswith('.h5')]
tm_labels_file = [f for f in uploaded.keys() if f.lower() == 'labels.txt']

print(f"\nImages: {len(image_files)}")
print(f"CSV:    {len(csv_files)} — {csv_files}")
if tm_model_file:
    print(f"Teachable Machine model: {tm_model_file[0]}")
    print(f"Teachable Machine labels: {tm_labels_file[0] if tm_labels_file else 'NOT FOUND — needed with .h5'}")
else:
    print(f"Teachable Machine model: not uploaded (YOLO-only mode)")

## Step 3: Parse GPS coordinates from CSV

The CSV has multiple rows per waypoint (25Hz sampling). We average `lat`/`lon` per waypoint to get one coordinate per image.

In [ ]:
import pandas as pd

df_trail = pd.read_csv(csv_files[0])
print(f"Loaded {csv_files[0]}: {len(df_trail)} rows, {df_trail['wp'].nunique()} waypoints")
print(f"Columns: {list(df_trail.columns)}")

# Handle both 'lon' and 'lng' column naming conventions
lon_col = 'lon' if 'lon' in df_trail.columns else 'lng'

# Average lat/lon per waypoint
wp_coords = df_trail.groupby('wp').agg(
    lat=('lat', 'mean'),
    lon=(lon_col, 'mean')
).reset_index()

print(f"\n{len(wp_coords)} waypoints with averaged coordinates")
wp_coords.head()

## Step 4: Load YOLOv8 model

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

# COCO classes of interest for trailside amenities
TARGET_CLASSES = {
    13: 'bench',
    56: 'chair',
    11: 'stop sign',
    58: 'potted plant',
}

CONF_THRESHOLD = 0.25

print(f"YOLOv8n loaded")
print(f"Target classes: {TARGET_CLASSES}")
print(f"Confidence threshold: {CONF_THRESHOLD}")

## Step 5: Load Teachable Machine model (if uploaded)

Teachable Machine exports a Keras model that classifies 224×224 images. We'll use it as a second-pass classifier on each YOLO crop.

In [ ]:
import numpy as np

tm_model = None
tm_labels = None

if tm_model_file and tm_labels_file:
    from tensorflow.keras.models import load_model

    tm_model = load_model(tm_model_file[0], compile=False)

    with open(tm_labels_file[0], 'r') as f:
        tm_labels = [line.strip().split(' ', 1)[-1] for line in f.readlines() if line.strip()]

    print(f"Teachable Machine model loaded: {tm_model_file[0]}")
    print(f"Input shape: {tm_model.input_shape}")
    print(f"Classes ({len(tm_labels)}): {tm_labels}")
else:
    print("No Teachable Machine model — skipping custom classification")

## Step 6: Run detection on all images

YOLO detects objects with bounding boxes. If a Teachable Machine model is loaded, each crop is classified to get a specific amenity type.

In [ ]:
import re
from PIL import Image

def classify_crop_tm(img, box_xyxy, tm_model, tm_labels):
    """Crop a detection from the image and classify with Teachable Machine."""
    x1, y1, x2, y2 = [int(v) for v in box_xyxy]
    crop = img.crop((x1, y1, x2, y2)).resize((224, 224))
    arr = np.array(crop, dtype=np.float32) / 127.5 - 1.0  # TM normalization: [-1, 1]
    arr = np.expand_dims(arr, axis=0)
    preds = tm_model.predict(arr, verbose=0)[0]
    best_idx = np.argmax(preds)
    return tm_labels[best_idx], float(preds[best_idx])

detections = []
yolo_results = {}  # store results for visualization

for fname in image_files:
    img = Image.open(fname).convert('RGB')
    img_w, img_h = img.size
    img_area = img_w * img_h

    # Run YOLO inference
    results = model(fname, conf=CONF_THRESHOLD, verbose=False)
    result = results[0]
    yolo_results[fname] = result

    # Extract waypoint number from filename (img_XXXX.jpg → XXXX)
    match = re.search(r'(\d+)', fname)
    wp_num = int(match.group(1)) if match else None

    # Look up averaged GPS coordinates
    lat, lon = None, None
    if wp_num is not None:
        wp_row = wp_coords[wp_coords['wp'] == wp_num]
        if not wp_row.empty:
            lat = wp_row.iloc[0]['lat']
            lon = wp_row.iloc[0]['lon']

    # Process each detection box
    for box in result.boxes:
        cls_id = int(box.cls[0])
        if cls_id not in TARGET_CLASSES:
            continue

        conf = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        bbox_area = (x2 - x1) * (y2 - y1)
        bbox_area_pct = (bbox_area / img_area) * 100

        row = {
            'filename': fname,
            'waypoint': wp_num,
            'lat': lat,
            'lon': lon,
            'class': TARGET_CLASSES[cls_id],
            'confidence': round(conf, 4),
            'bbox_x1': round(x1, 1),
            'bbox_y1': round(y1, 1),
            'bbox_x2': round(x2, 1),
            'bbox_y2': round(y2, 1),
            'bbox_area_pct': round(bbox_area_pct, 2),
        }

        # Teachable Machine classification on the crop
        if tm_model is not None:
            tm_class, tm_conf = classify_crop_tm(img, box.xyxy[0].tolist(), tm_model, tm_labels)
            row['tm_class'] = tm_class
            row['tm_confidence'] = round(tm_conf, 4)

        detections.append(row)

    n_det = sum(1 for box in result.boxes if int(box.cls[0]) in TARGET_CLASSES)
    if n_det > 0:
        print(f"{fname}: {n_det} detection(s)")

print(f"\nDone. {len(detections)} total detections across {len(image_files)} images.")

## Step 7: Build output CSV

In [ ]:
columns = [
    'filename', 'waypoint', 'lat', 'lon', 'class', 'confidence',
    'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2', 'bbox_area_pct'
]
if tm_model is not None:
    columns += ['tm_class', 'tm_confidence']

df_det = pd.DataFrame(detections, columns=columns)
df_det

## Step 8: Summary

In [ ]:
images_with_det = df_det['filename'].nunique() if len(df_det) > 0 else 0

print(f"Total images processed: {len(image_files)}")
print(f"Images with detections: {images_with_det}")
print(f"Total detections:       {len(df_det)}")
print()
if len(df_det) > 0:
    print("Detections per YOLO class:")
    for cls_name, count in df_det['class'].value_counts().items():
        print(f"  {cls_name}: {count}")

    if tm_model is not None:
        print()
        print("Detections per Teachable Machine class:")
        for cls_name, count in df_det['tm_class'].value_counts().items():
            avg_conf = df_det[df_det['tm_class'] == cls_name]['tm_confidence'].mean()
            print(f"  {cls_name}: {count}  (avg confidence {avg_conf:.2f})")

## Step 9: Visualize detections

Grid of the first 20 images that have detections, with bounding boxes drawn.

In [ ]:
import matplotlib.pyplot as plt

# Get filenames that have at least one target detection
files_with_det = [f for f in image_files
                  if any(int(box.cls[0]) in TARGET_CLASSES for box in yolo_results[f].boxes)]

show_files = files_with_det[:20]
n = len(show_files)

if n == 0:
    print("No target-class detections to visualize.")
else:
    cols = min(4, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
    if rows == 1 and cols == 1:
        axes = [[axes]]
    elif rows == 1:
        axes = [axes]
    elif cols == 1:
        axes = [[ax] for ax in axes]

    for idx, fname in enumerate(show_files):
        r, c = divmod(idx, cols)
        ax = axes[r][c]

        # Use ultralytics built-in plotting
        result = yolo_results[fname]
        plotted = result.plot()
        # plotted is BGR numpy array, convert to RGB
        ax.imshow(plotted[:, :, ::-1])

        # Add TM labels as subtitle if available
        title = fname
        if tm_model is not None:
            tm_hits = df_det[df_det['filename'] == fname]['tm_class'].tolist()
            if tm_hits:
                title += f"\nTM: {', '.join(tm_hits)}"
        ax.set_title(title, fontsize=9)
        ax.axis('off')

    # Hide unused subplots
    for idx in range(n, rows * cols):
        r, c = divmod(idx, cols)
        axes[r][c].axis('off')

    plt.tight_layout()
    plt.show()

## Step 10: Export and download

In [ ]:
output_path = 'trail_detections.csv'
df_det.to_csv(output_path, index=False)
print(f"Saved {output_path} ({len(df_det)} rows)")

files.download(output_path)